In [39]:
from spc import Rules
from config import testDf
import numpy as np


In [ ]:
df = testDf.copy()
threshold = 2

# Flag alternating points
df["diff"] = df["value"] - df["value"].shift()

def increasing_or_decreasing_int(x): 
    if x > 0: 
        return 1
    elif x < 0: 
        return -1
    else: 
        return 0 
    
df["increasing or decreasing"] = df["diff"].apply(increasing_or_decreasing_int)

df["alternating"] = df["increasing or decreasing"] + df["increasing or decreasing"].shift(1)

df["alternating tf"] = df["alternating"].apply(lambda x: True if x == 0 else False)

# # 1. Start of streak points: 
df["start of streak"] = df["alternating tf"].ne(df["alternating tf"].shift(1))

def start_of_alternating_streak(alternating:float, start_of_streak:bool): 
    if alternating != 0: 
        return False
    if start_of_streak == True and alternating == 0: 
        return True
    else: 
        return False

df["start of alternating streak"] = df.apply(lambda x: start_of_alternating_streak(x["alternating"], x["start of streak"]), axis=1)

# 2) Create a unique idenfifier which increments one every time a new streak begins
df["streak_id"] = df["start of alternating streak"].cumsum()

# 2a) Everywhere where the points are not alternating, set that streak_id to nothing to prevent false flags
df.loc[df["alternating"] != 0, "streak_id"] = np.nan


# 3) Groupby "streak_id" and cumulative count occurrances of streak_id and add 1 since count starts at zero 
df["streak_counter"] = df.groupby("streak_id").cumcount() + 1

# 4) Apply rule for any points of streak counter which are greater than threshold
df["rule2v"] = df["streak_counter"].apply(lambda x: True if x >=  threshold else False)

df

,user_id,created,sn,feature_id,machine,value,diff,increasing or decreasing,alternating,alternating tf,start of streak,start of alternating streak,streak_id,streak_counter,rule2v
id,,,,,,,,,,,,,,,
1,1,2026-01-30 06:16:58.274429,123-abc,1,drill,9.770483,NaN,0,NaN,False,True,False,NaN,NaN,False
2,1,2026-01-30 06:16:58.274429,123-abc,1,drill,11.502786,1.732303,1,1.0,False,False,False,NaN,NaN,False
3,1,2026-01-30 06:16:58.274429,123-abc,1,drill,12.705915,1.203128,1,2.0,False,False,False,NaN,NaN,False
4,1,2026-01-30 06:16:58.274429,123-abc,1,drill,13.859887,1.153972,1,2.0,False,False,False,NaN,NaN,False
5,1,2026-01-30 06:16:58.274429,123-abc,1,drill,14.163517,0.303631,1,2.0,False,False,False,NaN,NaN,False
6,1,2026-01-30 06:16:58.274429,123-abc,1,drill,14.513122,0.349604,1,2.0,False,False,False,NaN,NaN,False
7,1,2026-01-30 06:16:58.274429,123-abc,1,drill,6.888219,-7.624902,-1,0.0,True,True,True,1.0,1.0,False
8,1,2026-01-30 06:16:58.274429,123-abc,1,drill,7.036454,0.148235,1,0.0,True,False,False,1.0,2.0,True
9,1,2026-01-30 06:16:58.274429,123-abc,1,drill,7.736955,0.700501,1,2.0,False,True,False,NaN,NaN,False


In [ ]:
jkkkdf.columns

Index(['user_id', 'created', 'sn', 'feature_id', 'machine', 'value', 'diff',
       'increasing or decreasing', 'alternating', 'alternating tf'],
      dtype='object')